In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn

MODEL_PATH = 'line_follower.pth'
IMG_SIZE   = 224
CROP_FROM  = 0  # crop top 25% of frame — increase to crop more

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on: {device}')

class CropTop(object):
    """Removes the top CROP_FROM fraction of the frame."""
    def __call__(self, img):
        w, h = img.size
        return img.crop((0, int(h * CROP_FROM), w, h))

model = torchvision.models.mobilenet_v2(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 1),
    nn.Tanh()
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=False))
model = model.to(device)
model.eval()
print('Model loaded successfully.')

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    CropTop(),                    # removes top 25% before inference
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [2]:
import traitlets
import cv2
import numpy as np
import pyzed.sl as sl
import threading
import motors
from traitlets.config.configurable import SingletonConfigurable

class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super(Camera, self).__init__()
        self.zed = sl.Camera()
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA
        init_params.camera_fps = 100
        init_params.depth_mode = sl.DEPTH_MODE.PERFORMANCE  # enabled for corner anticipation
        init_params.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print('Camera Open:', repr(status))
            self.zed.close()
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        camera_info = self.zed.get_camera_information()
        self.width   = camera_info.camera_configuration.resolution.width
        self.height  = camera_info.camera_configuration.resolution.height
        self.image   = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)
        self.depth   = sl.Mat(self.width, self.height, sl.MAT_TYPE.F32_C1, sl.MEM.CPU)
        self.depth_image = None

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                self.zed.retrieve_measure(self.depth, sl.MEASURE.DEPTH)
                bgra = self.image.get_data()
                self.color_value = cv2.cvtColor(bgra, cv2.COLOR_BGRA2BGR)
                self.depth_image = np.asanyarray(self.depth.get_data())

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()

def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

camera = Camera()
camera.start()
robot = motors.MotorsYukon(mecanum=False)
print('Camera and robot ready.')

[2026-05-08 08:29:08 UTC][ZED][INFO] Logging level INFO
[2026-05-08 08:29:08 UTC][ZED][INFO] Logging level INFO
[2026-05-08 08:29:08 UTC][ZED][INFO] Logging level INFO
Camera and robot ready.
[2026-05-08 08:29:09 UTC][ZED][INFO] [Init]  Depth mode: PERFORMANCE
[2026-05-08 08:29:10 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2026-05-08 08:29:10 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2026-05-08 08:29:10 UTC][ZED][INFO] [Init]  Video mode: VGA@100
[2026-05-08 08:29:10 UTC][ZED][INFO] [Init]  Serial Number: S/N 37200128


In [3]:
import ipywidgets as widgets
from IPython.display import display
from collections import deque
import numpy as np

# Display widgets
display_feed = widgets.Image(format='jpeg', width='45%')
display_mask = widgets.Image(format='jpeg', width='45%')
status_label = widgets.Label(value='Status: Starting...')
layout = widgets.Layout(width='100%')
display(widgets.VBox([
    widgets.HBox([display_feed, display_mask], layout=layout),
    status_label
]))

YELLOW_LOWER = np.array([20, 100, 100])
YELLOW_UPPER = np.array([35, 255, 255])

# Speed parameters
SPEED_MAX    = 0.75
SPEED_MIN    = 0.20
SPEED_SEARCH = 0.30
LOST_FRAMES_MAX = 15

# Steering parameters
STEERING_DEADZONE = 0.3  # reduced from 0.20
STEERING_GAIN     = 0.6

ANTICIPATION_WINDOW    = 12
ANTICIPATION_THRESHOLD = 0.07

DEPTH_BRAKE_START = 200
DEPTH_BRAKE_STOP  = 100

steering_history  = deque(maxlen=2)
anticipation_buf  = deque(maxlen=ANTICIPATION_WINDOW)
lost_frame_count  = 0
last_steering     = 0.0
frame_count       = 0

def predict_steering(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    x   = preprocess(rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(x)
    return out.item()

def yellow_mask_debug(frame):
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, YELLOW_LOWER, YELLOW_UPPER)
    return cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)

def get_depth_brake_factor():
    depth_image = camera.depth_image if hasattr(camera, 'depth_image') else None
    if depth_image is None:
        return 0.0
    h, w  = depth_image.shape
    strip = depth_image[int(h * 0.5):int(h * 0.85), int(w * 0.25):int(w * 0.75)]
    strip = np.nan_to_num(strip, nan=0.0).astype(np.float32)
    strip[strip < 100]  = 0
    strip[strip > 4000] = 0
    if strip[strip != 0].size == 0:
        return 0.0
    min_dist = strip[strip != 0].min()
    if min_dist >= DEPTH_BRAKE_START:
        return 0.0
    if min_dist <= DEPTH_BRAKE_STOP:
        return 1.0
    return 1.0 - (min_dist - DEPTH_BRAKE_STOP) / (DEPTH_BRAKE_START - DEPTH_BRAKE_STOP)

def get_anticipation_factor():
    if len(anticipation_buf) < ANTICIPATION_WINDOW:
        return 0.0
    values = list(anticipation_buf)
    mid        = ANTICIPATION_WINDOW // 2
    early_avg  = sum(abs(v) for v in values[:mid])  / mid
    recent_avg = sum(abs(v) for v in values[mid:])  / (ANTICIPATION_WINDOW - mid)
    trend      = recent_avg - early_avg
    if trend < ANTICIPATION_THRESHOLD:
        return 0.0
    return min(1.0, (trend - ANTICIPATION_THRESHOLD) / 0.3)

def compute_forward_speed(steering, depth_brake, anticipation_brake):
    steering_brake = min(1.0, abs(steering) * 0.85)
    total_brake = max(steering_brake, depth_brake, anticipation_brake)
    speed = SPEED_MAX - (SPEED_MAX - SPEED_MIN) * total_brake
    return round(max(SPEED_MIN, speed), 3)

def draw_deadzone(img, steering, deadzone):
    h, w   = img.shape[:2]
    overlay = img.copy()

    # Deadzone band spans the full height of the frame
    dz_left  = int((0.5 - deadzone / 2) * w)
    dz_right = int((0.5 + deadzone / 2) * w)

    inside = abs(steering) < deadzone
    colour = (0, 200, 0) if inside else (0, 200, 200)  # green inside, yellow outside

    # Draw filled semi-transparent rectangle
    cv2.rectangle(overlay, (dz_left, 0), (dz_right, h), colour, -1)
    cv2.addWeighted(overlay, 0.15, img, 0.85, 0, img)  # 15% opacity

    # Draw solid border lines on deadzone edges
    cv2.line(img, (dz_left,  0), (dz_left,  h), colour, 2)
    cv2.line(img, (dz_right, 0), (dz_right, h), colour, 2)

    # Label
    cv2.putText(img, 'DZ', (dz_left + 4, 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, colour, 1)

def on_frame(change):
    global lost_frame_count, last_steering, frame_count

    frame = change['new']
    if frame is None:
        return

    frame_count += 1
    h, w = frame.shape[:2]
    
    raw_steering = predict_steering(frame)

    steering_history.append(raw_steering)
    steering = sum(steering_history) / len(steering_history)

    anticipation_buf.append(raw_steering)

    if abs(raw_steering) > 0.90:
        lost_frame_count += 1
    else:
        lost_frame_count = 0
        last_steering    = steering

    line_lost = lost_frame_count >= LOST_FRAMES_MAX

    depth_brake        = get_depth_brake_factor()
    anticipation_brake = get_anticipation_factor()
    fwd_speed          = compute_forward_speed(steering, depth_brake, anticipation_brake)

    if anticipation_brake > 0.1 and anticipation_brake >= depth_brake:
        brake_source = f'ANTICIPATING ({anticipation_brake:.2f})'
    elif depth_brake > 0.1:
        brake_source = f'DEPTH BRAKE ({depth_brake:.2f})'
    else:
        brake_source = ''

    if line_lost:
        if last_steering < 0:
            robot.left(SPEED_SEARCH)
        else:
            robot.right(SPEED_SEARCH)
        status = f'SEARCHING  lost={lost_frame_count}f'

    elif abs(steering) < STEERING_DEADZONE:
        robot.forward(fwd_speed)
        status = f'FORWARD {fwd_speed:.2f}  steer={steering:.2f}  {brake_source}'

    elif steering < 0:
        turn_speed = round(STEERING_GAIN * abs(steering), 3)
        robot.left(turn_speed)
        status = f'LEFT {turn_speed:.2f}  steer={steering:.2f}  {brake_source}'

    else:
        turn_speed = round(STEERING_GAIN * abs(steering), 3)
        robot.right(turn_speed)
        status = f'RIGHT {turn_speed:.2f}  steer={steering:.2f}  {brake_source}'

    status_label.value = f'Status: {status}'

    # Display every 5th frame
    if frame_count % 2 == 0:
        annotated = frame.copy()
        # Draw crop line — shows what the model can and cannot see
        crop_y = int(h * CROP_FROM)
        cv2.line(annotated, (0, crop_y), (w, crop_y), (255, 0, 255), 2)
        cv2.putText(annotated, 'model sees below here', (5, crop_y - 5),
        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)
        h, w      = annotated.shape[:2]
        colour    = (0, 255, 0) if not line_lost else (0, 0, 255)

        # Draw deadzone as shaded band across full frame
        draw_deadzone(annotated, steering, STEERING_DEADZONE)

        # Steering bar at bottom
        bar_centre = w // 2
        bar_end    = int(bar_centre + steering * (w // 3))
        cv2.line(annotated, (bar_centre, h - 10), (bar_end, h - 10), colour, 4)
        cv2.circle(annotated, (bar_centre, h - 10), 4, (255, 255, 255), -1)

        # Braking indicator bar on right edge
        total_brake  = max(min(1.0, abs(steering) * 0.85), depth_brake, anticipation_brake)
        brake_height = int(total_brake * h)
        brake_colour = (0, 165, 255) if total_brake > 0.3 else (0, 255, 0)
        cv2.rectangle(annotated, (w - 15, h - brake_height), (w - 5, h), brake_colour, -1)
        cv2.putText(annotated, 'BRK', (w - 20, h - brake_height - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, brake_colour, 1)

        if anticipation_brake > 0.1:
            cv2.putText(annotated, 'CORNER AHEAD',
                        (10, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)

        cv2.putText(annotated, f'steer: {steering:.2f}',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, colour, 2)
        cv2.putText(annotated, f'speed: {fwd_speed:.2f}',
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, colour, 2)

        scale = 0.3
        display_feed.value = bgr8_to_jpeg(
            cv2.resize(annotated, None, fx=scale, fy=scale))
        display_mask.value = bgr8_to_jpeg(
            cv2.resize(yellow_mask_debug(frame), None, fx=scale, fy=scale))

camera.observe(on_frame, names=['color_value'])
print('Line follower running. Execute the cell below to stop.')

Line follower running. Execute the cell below to stop.


In [4]:
camera.unobserve(on_frame, names=['color_value'])
robot.stop()
print('Robot stopped.')

Robot stopped.


## Fine-Tuning Tips

| Problem | Fix |
|---|---|
| Brakes too early before corners | Increase `ANTICIPATION_THRESHOLD` or reduce `DEPTH_BRAKE_START` |
| Doesn't slow down enough | Reduce `SPEED_MIN` or lower `ANTICIPATION_THRESHOLD` |
| Jittery on straights | Widen `STEERING_DEADZONE` |
| Misses sharp corners | Reduce `STEERING_GAIN`, collect more corner data |
| Spins when line lost | Reduce `LOST_FRAMES_MAX` |

In [ ]:
import shutil
import os

shutil.make_archive('dataset_backup', 'zip', '.', 'dataset')
print(f"Done! Size: {os.path.getsize('dataset_backup.zip') / 1024 / 1024:.1f} MB")